# Session 2: Grouping & Aggregations — Transactions and Products

## What This Notebook Does

In the loading and inspection session we understood the structure of the data.
This session moves into actual analysis. The goal is to group `transactions_clean.csv`
by product and time, and merge it with `products.csv` to answer real business questions.

---

## Questions Covered

**Sales Volume**
- Which product sold the most units and which sold the least?
- Which product-month combination had the single highest sales volume?

**Revenue & Profit**
- Does the highest revenue product also have the highest units sold — or are they different?
- Which product carries the most gross profit weight for the business?
- Which product has the best gross margin ratio and which has the worst?

**Demand Stability**
- Which product has the biggest gap between its best and worst sales day?
- What does that tell us about forecasting difficulty?

**Seasonality**
- Which month generated the highest revenue and which was the weakest?

**Category & Base Demand**
- Which product category drives the most revenue and daily profit?
- Does a higher base demand actually lead to higher real revenue?

---

## Files Used
- `transactions_clean.csv` — daily sales, revenue, and profit per product
- `products.csv` — product reference table with cost, price, category, and base demand

In [15]:
# importing essential libraries
import pandas as pd
import numpy as np

In [16]:
# loading the datasets
transactions_data = pd.read_csv('../../data/transactions_clean.csv')
products_data = pd.read_csv('../../data/products.csv')

In [17]:
transactions_data.head(6)

,date,product_id,units_sold,revenue,cogs,gross_profit
0,2024-01-01,P001,15,1349.85,675.0,674.85
1,2024-01-01,P002,16,479.84,192.0,287.84
2,2024-01-01,P003,17,339.83,136.0,203.83
3,2024-01-01,P004,12,719.88,360.0,359.88
4,2024-01-01,P005,5,649.95,275.0,374.95
5,2024-01-02,P001,14,1259.86,630.0,629.86


In [18]:
units_sold_summary = transactions_data.groupby('product_id')['units_sold'].sum().reset_index().sort_values(ascending=False, by='units_sold')
units_sold_summary

,product_id,units_sold
2,P003,12816
0,P001,9598
1,P002,7616
3,P004,6503
4,P005,4224


In [19]:
most_sold = units_sold_summary.iloc[0]
least_sold = units_sold_summary.iloc[-1]
print(f"Product that sold the MOST units: {most_sold['product_id']} ({most_sold['units_sold']} units)")
print(f"Product that sold the LEAST units: {least_sold['product_id']} ({least_sold['units_sold']} units)")

Product that sold the MOST units: P003 (12816 units)
Product that sold the LEAST units: P005 (4224 units)


In [20]:
revenue_summary = transactions_data.groupby('product_id')['revenue'].sum().reset_index().sort_values(ascending=False, by='revenue')
revenue_summary

,product_id,revenue
0,P001,863724.02
4,P005,549077.76
3,P004,390114.97
2,P003,256191.84
1,P002,228403.84


### Does The highest revenue product have the highest units sold?

No they are different products. The highest revenue product is P001 ($863,724.02). And the 
highest units sold product is P003 (12816)

In [21]:
# Comparison between revenue and units_sold
product_comparison = transactions_data.groupby('product_id')[['units_sold', 'revenue']].sum().reset_index()

product_comparison['price_per_unit'] = product_comparison['revenue'] / product_comparison['units_sold']

product_comparison = product_comparison.sort_values(by='revenue', ascending=False)
product_comparison

,product_id,units_sold,revenue,price_per_unit
0,P001,9598,863724.02,89.99
4,P005,4224,549077.76,129.99
3,P004,6503,390114.97,59.99
2,P003,12816,256191.84,19.99
1,P002,7616,228403.84,29.99


In [22]:
gross_profit_summary = transactions_data.groupby('product_id')['gross_profit'].sum().reset_index()
gross_profit_summary['gross_share_pct'] = (gross_profit_summary['gross_profit'] / gross_profit_summary['gross_profit'].sum()) * 100
gross_profit_summary

,product_id,gross_profit,gross_share_pct
0,P001,431814.02,34.985309
1,P002,137011.84,11.100616
2,P003,153663.84,12.449751
3,P004,195024.97,15.800804
4,P005,316757.76,25.663521


### Analysis: Gross Profit Share by Product

To determine which product carries the most financial weight for the business, we calculated each product's total gross profit and its percentage share of the company's total gross profit ($\sum \text{Gross Profit}$).

| Product ID | Total Gross Profit | Profit Share (%) | Role in Business |
| :--- | :--- | :--- | :--- |
| **P001** | \$431,814.02 | **34.99%** | **Primary Driver:** Carries over a third of the business's profit weight. |
| **P005** | \$316,757.76 | **25.66%** | **Strong Contender:** Second highest profit contributor. |
| **P004** | \$195,024.97 | 15.80% | Moderate contributor. |
| **P003** | \$153,663.84 | 12.45% | Low profit weight (High volume, but low margin). |
| **P002** | \$137,011.84 | 11.10% | Lowest profit contributor. |

#### Key Takeaway
**P001** is single-handedly carrying the most profit weight for the business, accounting for **34.99%** of total gross profits. Combined with P005, these two premium products generate over **60%** of the company's entire profit margins, making them the most critical products to keep in stock.

In [23]:
units_sold_info = transactions_data.groupby('product_id')['units_sold'].agg(['mean', 'max', 'min'])
units_sold_info

,mean,max,min
product_id,,,
P001,26.295890,57,7
P002,20.865753,45,5
P003,35.112329,72,14
P004,17.816438,41,6
P005,11.572603,25,4


In [24]:
units_sold_info['sales_gap'] = units_sold_info['max'] - units_sold_info['min']

units_sold_info = units_sold_info.sort_values(by='sales_gap', ascending=False)
units_sold_info

,mean,max,min,sales_gap
product_id,,,,
P003,35.112329,72,14,58
P001,26.295890,57,7,50
P002,20.865753,45,5,40
P004,17.816438,41,6,35
P005,11.572603,25,4,21


### Analysis: Daily Sales Variability & Demand Stability

To analyze the stability of customer demand, we evaluated the statistical range (the "gap") between the worst and best sales days for each product using a single `.agg()` call.

| Product ID | Mean Daily Sales | Min Daily Sales | Max Daily Sales | Sales Gap (Max - Min) | Demand Stability |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **P003** | 35.11 | 14 | 72 | **58** | **Highly Volatile** (Low Stability) |
| **P001** | 26.30 | 7 | 57 | 50 | Volatile |
| **P002** | 20.87 | 5 | 45 | 40 | Moderate |
| **P004** | 17.82 | 6 | 41 | 35 | Moderate |
| **P005** | 11.57 | 4 | 25 | 21 | Stable (Consistent Low Volume) |

#### Key Takeaway
**P003** exhibits the largest daily sales variance with a **58-unit gap** between its minimum (14) and maximum (72) sales performance. 

While P003 has the highest average daily sales volume (~35 units), its massive swings demonstrate **low demand stability**. From a supply chain perspective, this product is the most difficult to forecast and carries the highest risk of sudden stockouts during unexpected peak demand spikes.

In [33]:
transactions_data['date'] = pd.to_datetime(transactions_data['date'])

transactions_data['month'] = transactions_data['date'].dt.strftime('%B')

monthly_revenue = transactions_data.groupby('month')['revenue'].sum().reset_index().sort_values(by='revenue', ascending=False)
monthly_revenue

,month,revenue
2,December,280250.59
9,November,229949.44
5,July,215031.67
6,June,204083.50
1,August,199224.05
8,May,185246.89
10,October,184757.37
7,March,172159.25
0,April,171459.50
11,September,169709.57


### Analysis: Monthly Revenue Trends

To evaluate seasonal performance across the full year, we transformed the transaction dates into a monthly format and aggregated total combined revenue. 

| Month | Total Revenue | Performance Level |
| :--- | :--- | :--- |
| **December** | \$280,250.59 | **Highest Revenue Month** |
| **November** | \$229,949.44 | High Performance |
| **July** | \$215,031.67 | High Performance |
| **June** | \$204,083.50 | Mid-tier |
| **August** | \$199,224.05 | Mid-tier |
| **May** | \$185,246.89 | Mid-tier |
| **October** | \$184,757.37 | Mid-tier |
| **March** | \$172,159.25 | Mid-tier |
| **April** | \$171,459.50 | Mid-tier |
| **September** | \$169,709.57 | Mid-tier |
| **February** | \$146,753.87 | Low Performance |
| **January** | \$128,886.73 | **Weakest Revenue Month** |

#### Key Takeaway
The business experienced its peak performance in **December**, driving a massive **\$280,250.59** in revenue (likely due to holiday shopping and year-end demand). Conversely, **January** was the slowest period for the company, bringing in only **\$128,886.73**. This suggests a classic post-holiday seasonal downturn, indicating that the business might want to run special promotions or targeted clearing sales at the start of the year to lift lagging winter numbers.

In [26]:
units_sold_per_month_id = transactions_data.groupby(['month', 'product_id'], observed=True)['units_sold'].sum().reset_index()
units_sold_per_month_id = units_sold_per_month_id.sort_values(by='units_sold', ascending=False)
units_sold_per_month_id

,month,product_id,units_sold
12,December,P003,1523
47,November,P003,1296
10,December,P001,1217
27,July,P003,1153
32,June,P003,1151
7,August,P003,1146
42,May,P003,1043
52,October,P003,1037
2,April,P003,984
57,September,P003,964


### Analysis: Peak Product-Month Sales Volume

To pinpoint the specific peak demand periods for individual products, we analyzed the total units sold by grouping the transaction data by both `month` and `product_id` simultaneously.

| Rank | Month | Product ID | Total Units Sold | Performance Note |
| :---: | :--- | :--- | :---: | :--- |
| **1** | **December** | **P003** | **1,523** | **Highest Peak Volume of the Year** |
| 2 | November | P003 | 1,296 | Strong Holiday Ramp-up |
| 3 | December | P001 | 1,217 | Premium Product Peak |
| 4 | July | P003 | 1,153 | Summer Demand Spike |
| 5 | June | P003 | 1,151 | Mid-year Peak |

#### Key Takeaway
**Product P003 in December** represents the absolute highest sales volume combination across the full year, moving **1,523 units** in a single month. 

This aligns with our previous findings: P003 is the business's highest-volume product overall, and December is the company's highest-revenue month. This insight tells supply chain managers that inventory levels for P003 must be aggressively increased ahead of Q4 to ensure fulfillment capabilities don't break during this massive seasonal surge.

In [27]:
merged_trans_prod = pd.merge(transactions_data, products_data, on='product_id')
merged_trans_prod.head()

,date,product_id,units_sold,revenue,cogs,gross_profit,month,name,category,unit_cost,unit_price,base_demand
0,2024-01-01,P001,15,1349.85,675.0,674.85,January,Wireless Headphones,Electronics,45.0,89.99,22
1,2024-01-01,P002,16,479.84,192.0,287.84,January,Yoga Mat,Fitness,12.0,29.99,18
2,2024-01-01,P003,17,339.83,136.0,203.83,January,Stainless Water Bottle,Kitchen,8.0,19.99,30
3,2024-01-01,P004,12,719.88,360.0,359.88,January,Bluetooth Speaker,Electronics,30.0,59.99,15
4,2024-01-01,P005,5,649.95,275.0,374.95,January,Winter Jacket,Apparel,55.0,129.99,10


In [28]:
daily_category_profit = merged_trans_prod.groupby(['category', 'date'])['gross_profit'].sum().reset_index()
avg_daily_profit = daily_category_profit.groupby('category')['gross_profit'].mean().reset_index(name='avg_daily_gross_profit')
total_category_revenue = merged_trans_prod.groupby('category')['revenue'].sum().reset_index(name='total_revenue')
category_analysis = pd.merge(total_category_revenue, avg_daily_profit, on='category').sort_values(by='total_revenue', ascending=False)
category_analysis

,category,total_revenue,avg_daily_gross_profit
1,Electronics,1253838.99,1717.367096
0,Apparel,549077.76,867.829479
3,Kitchen,256191.84,420.996822
2,Fitness,228403.84,375.374904


### Analysis: Category Revenue vs. Daily Profitability

To see if top-line revenue aligns with daily bottom-line performance, we calculated the total revenue per product category and compared it against the average daily gross profit.

| Category | Total Revenue | Avg. Daily Gross Profit | Profitability Rank |
| :--- | :--- | :--- | :---: |
| **Electronics** | \$1,253,838.99 | \$1,717.37 | **#1** (Highest) |
| **Apparel** | \$549,077.76 | \$867.83 | **#2** |
| **Kitchen** | \$256,191.84 | \$421.00 | **#3** |
| **Fitness** | \$228,403.84 | \$375.37 | **#4** (Lowest) |

#### Conclusion
**Yes, the highest revenue category is also the most profitable per day.** 
**Electronics** completely dominates the business, taking the top spot in both total revenue (1,253,838.99 usd) and daily gross profit (1,717.37 usd). In fact, the overall ranking remains perfectly identical for both metrics across all categories—meaning that higher sales volume and revenue translate directly into higher daily profit weights for this catalog.

In [29]:
gross_summary = transactions_data.groupby('product_id')[['revenue', 'gross_profit']].sum().reset_index()
gross_summary['gross_margin_pct'] = (gross_summary['gross_profit'] / gross_summary['revenue']) * 100
gross_summary = gross_summary.sort_values(ascending=False, by='gross_margin_pct')
gross_summary

,product_id,revenue,gross_profit,gross_margin_pct
1,P002,228403.84,137011.84,59.986662
2,P003,256191.84,153663.84,59.979990
4,P005,549077.76,316757.76,57.689053
0,P001,863724.02,431814.02,49.994444
3,P004,390114.97,195024.97,49.991665


### Analysis: Gross Profit Margin Ratio by Product

To measure how efficiently each product turns its sales revenue into profit, we calculated the Gross Margin Ratio ($\frac{\text{Total Gross Profit}}{\text{Total Revenue}} \times 100$) for each individual product.

| Product ID | Total Revenue | Total Gross Profit | Gross Margin Ratio (%) | Margin Rank |
| :--- | :--- | :--- | :---: | :---: |
| **P002** | \$228,403.84 | \$137,011.84 | **59.99%** | **#1 (Best)** |
| **P003** | \$256,191.84 | \$153,663.84 | 59.98% | #2 |
| **P005** | \$549,077.76 | \$316,757.76 | 57.69% | #3 |
| **P001** | \$863,724.02 | \$431,814.02 | 50.00% | #4 |
| **P004** | \$390,114.97 | \$195,024.97 | **49.99%** | **#5 (Worst)** |

#### Key Insights
* **The High-Efficiency Paradox:** **P002** has the **best profit margin (59.99%)** in the entire store, meaning it keeps nearly 60 cents of profit for every dollar spent. However, from our previous questions, we know P002 generated the *lowest overall revenue*. It is a highly efficient earner but suffers from lower total sales volume.
* **The Volume Leader Baseline:** Our highest revenue driver, **P001**, sits near the bottom with a **50.00% margin**. 
* **Healthy Catalog Standard:** Notably, even the "worst" margin product (**P004 at 49.99%**) is still capturing a massive 50% profit margin. In retail, keeping 50% to 60% of top-line revenue as gross profit across an entire product catalog indicates incredibly healthy pricing power and strong supplier agreements.

In [35]:
base_demand_summary = merged_trans_prod.groupby('product_id')['revenue'].sum().reset_index()
base_demand_ref = products_data[['product_id', 'base_demand']]
base_demand_summary = base_demand_summary.merge(base_demand_ref, on='product_id')
base_demand_summary

,product_id,revenue,base_demand
0,P001,863724.02,22
1,P002,228403.84,18
2,P003,256191.84,30
3,P004,390114.97,15
4,P005,549077.76,10


### Analysis: Base Demand vs. Actual Revenue Generated

To see if inherent product popularity (base demand) translates directly into financial success, we evaluated total revenue alongside the cumulative baseline demand for each product.

| Product ID | Base Demand | Total Revenue | Performance Type |
| :---: | :---: | :--- | :--- |
| **P003** | **30** *(Highest)* | \$256,191.84 | High Interest, Low Revenue Value |
| **P001** | 22 | \$863,724.02 | High Interest, **Highest Revenue Leader** |
| **P002** | 18 | \$228,403.84 | Moderate Interest, Lowest Revenue Value |
| **P004** | 15 | \$390,114.97 | Moderate Interest, Solid Revenue Performance |
| **P005** | **10** *(Lowest)* | **\$549,077.76** | Low Interest, **High Revenue Powerhouse** |

#### Conclusion
**No, a higher base demand does not guarantee higher revenue.** 
#### Why this happens:
This misalignment highlights the critical impact of **unit pricing** on top-line revenue:
* **The Premium Effect (P005):** P005 (Winter Jacket) has the absolute lowest baseline consumer demand in the dataset, but because it commands a premium price tag of **\$129.99**, it generates massive revenue chunks per individual transaction.
* **The Low-Cost Trap (P003):** P003 (Stainless Water Bottle) is incredibly popular and carries the highest structural demand, but its low unit price of **\$19.99** means that even moving massive volumes cannot match the revenue generation of the premium catalog items. 

